# M1 — EfficientNet-B0 (train/validation only)
Notebook gọi cùng mã nguồn với `experiments/run_m1.py`; chạy notebook **không đánh giá test**. Chỉ dùng test sau khi đã chốt backbone, augmentation, seed và ngưỡng trên validation.


In [ ]:
import os
import sys

# Nếu chạy Colab: mount Drive hoặc clone repository trước ô này.
current_dir = os.getcwd()
if os.path.exists(os.path.join(current_dir, 'data', 'manifest.csv')):
    PROJECT_ROOT = current_dir
elif os.path.exists(os.path.join(os.path.dirname(current_dir), 'data', 'manifest.csv')):
    PROJECT_ROOT = os.path.dirname(current_dir)
else:
    raise FileNotFoundError('Không tìm thấy data/manifest.csv; chuyển cwd vào repository')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)


## Huấn luyện và chọn mô hình
Preset `comparison`: RandomResizedCrop (scale 0.85–1.0, ratio 0.9–1.1), lật ngang/dọc, ColorJitter (0.1/0.1/0.1/0.02), xoay ±15°, chuẩn hóa ImageNet. Validation dùng LetterboxResize 224 và không augmentation. Checkpoint chọn theo Balanced Accuracy tại ngưỡng 0.5; sau đó chọn ngưỡng tối ưu Balanced Accuracy trên validation. Thêm seed vào danh sách sau khi đã chốt cấu hình.


In [ ]:
from experiments.run_m1 import run_m1_experiment

SEEDS = [42]  # Khi chạy thí nghiệm cuối: [42, 123, 2026]
EPOCHS = 10
AUGMENTATION_PRESET = 'comparison'
results_by_seed = {}
for seed in SEEDS:
    results_by_seed[seed] = run_m1_experiment(
        epochs=EPOCHS, batch_size=16, lr=1e-4, weight_decay=1e-2,
        seed=seed, augmentation_preset=AUGMENTATION_PRESET,
    )
print('Đã hoàn tất train/validation; chưa đánh giá test.')


## Learning curves (validation)


In [ ]:
import matplotlib.pyplot as plt

for seed, result in results_by_seed.items():
    history = result['history']
    epochs = [item['epoch'] for item in history]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, [item['train_loss'] for item in history], label='Train loss')
    axes[0].plot(epochs, [item['validation_loss'] for item in history], label='Validation loss')
    axes[0].set_title(f'Seed {seed}: loss')
    axes[0].legend()
    axes[1].plot(epochs, [item['train_balanced_accuracy'] for item in history], label='Train BAcc')
    axes[1].plot(epochs, [item['validation_balanced_accuracy_at_0.5'] for item in history], label='Validation BAcc @ 0.5')
    axes[1].axvline(result['best_epoch'], linestyle=':', color='green')
    axes[1].set_title(f'Seed {seed}: Balanced Accuracy')
    axes[1].legend()
    plt.tight_layout()
    plt.show()


## Tổng hợp validation qua các seed


In [ ]:
import numpy as np

metric_names = ['balanced_accuracy', 'f1_macro', 'sensitivity', 'roc_auc', 'pr_auc']
if len(results_by_seed) < 3:
    print('Chưa đủ 3 seeds để báo cáo mean ± SD chính thức.')
else:
    for name in metric_names:
        values = np.array([result['validation_metrics'][name] for result in results_by_seed.values()], dtype=float)
        print(f'{name}: {values.mean():.4f} ± {values.std(ddof=1):.4f}')


## Đánh giá test cuối cùng
Sau khi đã chốt cấu hình và chạy đủ các seed, dùng lệnh riêng (mỗi checkpoint chỉ đánh giá test sau khi lựa chọn đã cố định):

```bash
python3 experiments/run_m1.py --mode test --seed 42 --augmentation_preset comparison
```
Nếu chạy lại cùng seed, script sẽ từ chối ghi đè artifact cũ; chọn đường dẫn mới hoặc dùng `--overwrite` có chủ đích.
